# Two-state Markov regime-switching volatility experiment

The model uses 100 times log returns, a common constant conditional mean, regime-specific variances, and a two-state Markov transition matrix. Forecasting uses filtered probabilities at the current origin; smoothed probabilities are not used.

In [ ]:
from pathlib import Path
import pandas as pd
import yaml
from trm_forecast.evaluation.garch import evaluate_garch, evaluate_variance_benchmark
from trm_forecast.evaluation.protocol import PartitionSpec, TemporalEvaluationProtocol
from trm_forecast.evaluation.regime_switching import evaluate_regime_switching
from trm_forecast.models.garch import log_returns
from trm_forecast.models.regime_switching import MarkovSwitchingVolatility

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
config = yaml.safe_load((ROOT / 'configs/base.yaml').read_text())
frame = pd.read_csv(ROOT / 'data/interim/trm_normalized.csv', parse_dates=['date'])
series = frame.set_index('date')['trm'].sort_index()
evaluation = config['evaluation']
specs = tuple(PartitionSpec(name, values['start'], values['end']) for name, values in evaluation['partitions'].items())
result = TemporalEvaluationProtocol(evaluation['as_of_date'], tuple(evaluation['horizons']), specs).split(series)
regime = config['regime_switching']
garch = config['garch']
horizons = tuple(evaluation['horizons'])

## Training-only initial diagnostics

In [ ]:
train_returns = log_returns(result.partitions['train'], scale=regime['scale'])
training_model = MarkovSwitchingVolatility(maxiter=regime['maxiter']).fit(train_returns)
print('fit status:', training_model.fit_status)
print('parameters:', dict(zip(training_model.fitted_model.model.param_names, training_model.fitted_model.params)))
print('transition matrix:', training_model.fitted_model.regime_transition[:, :, 0])
print('last filtered probabilities:', training_model.filtered_probabilities(train_returns))
print('smoothed probabilities are intentionally not used for forecasting')

## Validation comparison

QLIKE is computed on the squared cumulative h-step scaled log return using the same origins and targets for the regime-switching model, GARCH, and the expanding historical-variance benchmark.

In [ ]:
validation_predictions, validation_metrics = evaluate_regime_switching(
    result, horizons, ('validation',), regime['min_history'], regime['scale'],
    regime['maxiter'], regime['refit_frequency']
)
_, validation_garch_metrics = evaluate_garch(
    result, horizons, ('validation',), garch['p'], garch['q'], garch['min_history'],
    garch['scale'], garch['maxiter'], garch['refit_frequency'], garch['confidence_level']
)
_, validation_benchmark_metrics = evaluate_variance_benchmark(result, horizons, ('validation',), regime['scale'])
validation_metrics.assign(model='regime_switching').merge(
    validation_garch_metrics.assign(model='garch'), on=['partition', 'horizon'], suffixes=('', '_garch')
)
validation_benchmark_metrics

## Frozen test comparison

The two-state specification is fixed before this cell. Test results are not used for model selection.

In [ ]:
test_predictions, test_metrics = evaluate_regime_switching(
    result, horizons, ('test',), regime['min_history'], regime['scale'],
    regime['maxiter'], regime['refit_frequency']
)
_, test_garch_metrics = evaluate_garch(
    result, horizons, ('test',), garch['p'], garch['q'], garch['min_history'],
    garch['scale'], garch['maxiter'], garch['refit_frequency'], garch['confidence_level']
)
_, test_benchmark_metrics = evaluate_variance_benchmark(result, horizons, ('test',), regime['scale'])
print(test_predictions.groupby('status').size())
test_metrics
test_benchmark_metrics